## Lorenz 96 path visualization

Loads trajectories from `datasets/LORENZ96/full_path/path_*.pt` (generated by `scripts/Lorenz96/generate_fields.py`) and plots selected state dimensions over time.

Each file is a tensor shaped `(T, 1, K, 1)` where:
- `T` = number of saved time frames (includes initial state at index 0)
- `K` = Lorenz 96 system dimension (number of grid points on the ring)

In [ ]:
import os

import matplotlib.pyplot as plt
import torch

from project_utils.paths import dataset_trajectory_dir

# --- user settings (edit these) ---
DATA_ROOT = "LORENZ96"
TRAJECTORY_SUBDIR = "full_path"

# Which path files to load (path_0.pt, path_1.pt, ...)
PATH_IDS = [0]

# Which state dimensions (grid indices 0 .. K-1) to plot
# For K=40, e.g. [0, 5, 10, 20, 39]; for K=1000, pick spread-out indices.
DIM_INDICES = [0, 100, 200, 300, 600]

# Optional: restrict time window for plots (None = use all frames)
T_START = None  # e.g. 0
T_END = None    # e.g. 500

# --- load paths ---
traj_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
paths = {}
for pid in PATH_IDS:
    fpath = os.path.join(traj_dir, f"path_{pid}.pt")
    traj = torch.load(fpath, map_location="cpu", weights_only=False)
    if traj.ndim == 4:
        traj = traj[:, 0, :, 0]  # (T, K)
    elif traj.ndim == 2:
        pass  # already (T, K)
    else:
        raise ValueError(f"Unexpected shape {tuple(traj.shape)} in {fpath}")

    t0 = 0 if T_START is None else int(T_START)
    t1 = traj.shape[0] if T_END is None else int(T_END)
    paths[pid] = traj[t0:t1]
    print(
        f"path_{pid}: file={fpath}\n"
        f"  shape={tuple(paths[pid].shape)}  dtype={paths[pid].dtype}  "
        f"min/max={paths[pid].min().item():.4g}/{paths[pid].max().item():.4g}"
    )

K = next(iter(paths.values())).shape[1]
valid_dims = [d for d in DIM_INDICES if 0 <= d < K]
if len(valid_dims) != len(DIM_INDICES):
    bad = [d for d in DIM_INDICES if d < 0 or d >= K]
    raise ValueError(f"DIM_INDICES out of range for K={K}: {bad}")
DIM_INDICES = valid_dims
print(f"K={K}, plotting dimensions: {DIM_INDICES}")

### Time series: selected dimensions, one subplot per path

In [ ]:
n_paths = len(paths)
fig, axs = plt.subplots(n_paths, 1, figsize=(10, 3 * n_paths), sharex=True, squeeze=False)

for row, (pid, traj) in enumerate(sorted(paths.items())):
    ax = axs[row, 0]
    t_axis = torch.arange(traj.shape[0])
    for d in DIM_INDICES:
        ax.plot(t_axis, traj[:, d].numpy(), label=f"x[{d}]", linewidth=1.0)
    ax.set_ylabel("x")
    ax.set_title(f"path_{pid}: selected dimensions")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", ncol=min(4, len(DIM_INDICES)), fontsize=8)

axs[-1, 0].set_xlabel("time index")
plt.tight_layout()
plt.show()

### Compare paths: one dimension across multiple trajectories

In [ ]:
# Pick one dimension to overlay across paths
COMPARE_DIM = DIM_INDICES[0]

fig, ax = plt.subplots(figsize=(10, 3.5))
for pid, traj in sorted(paths.items()):
    ax.plot(traj[:, COMPARE_DIM].numpy(), label=f"path_{pid}", linewidth=1.0, alpha=0.9)
ax.set_xlabel("time index")
ax.set_ylabel(f"x[{COMPARE_DIM}]")
ax.set_title(f"Dimension {COMPARE_DIM} across selected paths")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Spatial snapshot: full ring state at selected times

In [ ]:
# Ring state x_i for all i at a few time indices
SNAPSHOT_PATH = PATH_IDS[0]
SNAPSHOT_TIMES = [0, 100, 500, 1000]  # edit to fit your trajectory length

traj = paths[SNAPSHOT_PATH]
T = traj.shape[0]
times = [t for t in SNAPSHOT_TIMES if 0 <= t < T]
if not times:
    times = [0, T // 2, T - 1]

fig, axs = plt.subplots(len(times), 1, figsize=(10, 2.2 * len(times)), sharex=True, squeeze=False)
x_axis = torch.arange(K)

for row, t in enumerate(times):
    ax = axs[row, 0]
    ax.plot(x_axis, traj[t].numpy(), color="C0", linewidth=1.0)
    ax.set_ylabel("x")
    ax.set_title(f"path_{SNAPSHOT_PATH}, t={t}")
    ax.grid(True, alpha=0.3)

axs[-1, 0].set_xlabel("dimension index i")
plt.tight_layout()
plt.show()

### Space-time heatmap (time vs dimension)

In [ ]:
HEATMAP_PATH = PATH_IDS[0]
traj = paths[HEATMAP_PATH].numpy()  # (T, K)

# Optional downsample for large K / long trajectories
TIME_STRIDE = 1   # plot every N-th time frame
DIM_STRIDE = 1    # plot every N-th dimension

traj_ds = traj[::TIME_STRIDE, ::DIM_STRIDE]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(
    traj_ds.T,
    aspect="auto",
    origin="lower",
    cmap="RdBu_r",
    interpolation="nearest",
)
ax.set_xlabel(f"time index (stride={TIME_STRIDE})")
ax.set_ylabel(f"dimension index (stride={DIM_STRIDE})")
ax.set_title(f"path_{HEATMAP_PATH}: space-time heatmap")
fig.colorbar(im, ax=ax, label="x")
plt.tight_layout()
plt.show()